# Research Data Management with SageMaker Unified Studio (SMUS) 

This notebook only works with the [Research Computing Immersion Day](https://catalog.workshops.aws/research-computing) - "Research Data Management with SageMaker Unified Studio" module in an AWS led workshop, where certain resources are already provisioned. 


The "Getting Started" step needs to be done in AWS console manually. 
1. Create SageMaker Unified Studio domain
2. Update IAM Identity Center Configure - disable MFA
3. Reset admin/subscriber_marcuswilliams/publisher_sarahchen user passwords

This notebook is optional for the workshop. It helps you to provision/configure the following resources once you have completed the "Getting Started" step: 
1. Enroll admin/publisher_sarahchen/subscriber_marcuswilliams users in SMUS domain
2. Create "patient_cohort" project in SMUS domain and add admin user and publisher group as the owner of the project 
3. Create "patient_research" project in SMUS domain and add admin user and subscriber group as the owner of the project
4. Run crawlers to create glue data catalog tables 
5. Grant necessary lakeformation permissions for those tables


In [ ]:
!pip install boto3 --upgrade

In [ ]:
import boto3
from botocore.exceptions import ClientError

datazone = boto3.client('datazone')
identitystore = boto3.client('identitystore')
sso_admin = boto3.client('sso-admin')

instances = sso_admin.list_instances()
identity_store_id = instances['Instances'][0]['IdentityStoreId']

# Get domain ID (if you don't know it)
domains = datazone.list_domains()
domain_id=domains['items'][0]['id']

# Get project profile ID for "All capabilities"
profiles = datazone.list_project_profiles(domainIdentifier=domain_id)
all_capabilities_profile_id = None
sql_capabilities_profile_id = None

for profile in profiles['items']:
    if 'All capabilities' in profile.get('name', ''):
        all_capabilities_profile_id = profile['id']
    elif 'SQL analytics' in profile.get('name', ''):
        sql_capabilities_profile_id = profile['id']


login_url = f"https://{identity_store_id}.awsapps.com/start"
print(login_url)



In [ ]:
def get_project_id_by_name(datazone_client, d_id, project_name):
    # get sagemaker unified studio project_id by name
    projects = datazone_client.list_projects(domainIdentifier=d_id)
    project_id = None
    for project in projects['items']:
        if project['name'] == project_name:
            p_id = project['id']
            print("found the project id:", p_id)
            return p_id
            
def get_project_role_arn(datazone_client, d_id, p_id):
    # List environments and filter for tooling
    response = datazone.list_environments(
        domainIdentifier=d_id,
        projectIdentifier=p_id
    )
    
    tooling_env = None
    for env in response['items']:
        if 'tooling' in env['name'].lower() or env.get('provider') == 'TOOLING':
            tooling_env = env
            break
    
    response = datazone.get_environment(domainIdentifier = d_id, identifier=tooling_env['id'])
    
    project_role_arn = None
    for res in response['provisionedResources']:
        if res['name'] == 'userRoleArn':
            project_role_arn = res['value']
    
    return project_role_arn
    
def create_project(datazone_client, d_id, profile_id, project_name):
    try:
        # Create project
        response = datazone_client.create_project(
            domainIdentifier=d_id,
            name=project_name,
            projectProfileId=profile_id,
            description='SageMaker Unified Studio project with all capabilities'
        )        
        p_id = response['id']
        return p_id
    except ClientError as e:
        if e.response['Error']['Code'] == 'ConflictException':
            print(f"project already exists, ignore")
            # get the project_id with name
            p_id = get_project_id_by_name(datazone_client, d_id, project_name)
            return p_id
        else:
            raise   

def get_gid_by_name(identitystore_client, idstore_id, group_name):
    # Find group by name
    groups = identitystore_client.list_groups(IdentityStoreId=idstore_id)
    group_id = None
    for group in groups['Groups']:
        if group['DisplayName'] == group_name:
            group_id = group['GroupId']
            print("found the group id:", group_id)
            return group_id

def get_uid_by_name(identitystore_client, idstore_id, user_name):
    users = identitystore_client.list_users(IdentityStoreId=idstore_id)
    user_id = None
    for user in users['Users']:
        if user['UserName'] == user_name:
            user_id = user['UserId']
            print("found user id:", user_id) 
            return user_id



In [ ]:
def add_group_to_domain(datazone_client, d_id, g_id):
    # Add group to domain with domain_id
    try:
        response = datazone_client.create_group_profile(
            domainIdentifier=d_id,
            groupIdentifier=g_id
        )
        print(response)
    except ClientError as e:
        if e.response['Error']['Code'] == 'ValidationException':
            print(f"group already exists, ignore")
        else:
            raise
        

def add_user_to_domain(datazone_client, d_id, u_id):
    try:
        response = datazone_client.create_user_profile(
            domainIdentifier=domain_id,
            userIdentifier=u_id
        )
        print(response)
    except ClientError as e:
        if e.response['Error']['Code'] == 'ValidationException':
            print(f"user already exists, ignore")
        else:
            raise


# d_id: domain_id, p_id: project_id, m_id: member_id (groupd id or user id)
# type : GROUP or USER
# designation: 'PROJECT_CONTRIBUTOR' , 'PROJECT_OWNER'

def add_member_to_project(datazone_client, d_id, p_id, m_id, type, designation):    
    # Add group to project as Contributor
    member=None
    if type=='GROUP':
        member={'groupIdentifier': m_id}
    else:
        member={'userIdentifier': m_id}

    try:
        response = datazone_client.create_project_membership(
            domainIdentifier=d_id,
            projectIdentifier=p_id,
            member=member,
            designation=designation
        )
    except ClientError as e:
        if e.response['Error']['Code'] == 'ValidationException':
            print(f"user/group already exists in the project, ignore")
        else:
            raise    

## Data Management - Administration

### Add "admin" user and groups "publisher" and "subscriber" to SMUS domain

In [ ]:
publisher_gid = get_gid_by_name(identitystore, identity_store_id, "publisher")
subscriber_gid = get_gid_by_name(identitystore, identity_store_id, "subscriber")
admin_uid = get_uid_by_name(identitystore, identity_store_id, "admin") 

print( publisher_gid, subscriber_gid, admin_uid)

add_group_to_domain(datazone, domain_id, publisher_gid)
add_group_to_domain(datazone, domain_id, subscriber_gid)
add_user_to_domain(datazone, domain_id, admin_uid)



### Create projects

Create the "patien_cohort" project, and add admin and publisher group as the owner 

Create the "patient_research" project, and add admin and subscriber group as the onwer

In [ ]:

patient_cohort_project_id = create_project(datazone, domain_id, all_capabilities_profile_id, "patient_cohort")
add_member_to_project(datazone, domain_id, patient_cohort_project_id, publisher_gid, "GROUP", "PROJECT_OWNER")
add_member_to_project(datazone, domain_id, patient_cohort_project_id, admin_uid, "USER", "PROJECT_OWNER")

In [ ]:
patient_research_project_id = create_project(datazone, domain_id, all_capabilities_profile_id, "patient_research")
add_member_to_project(datazone, domain_id, patient_research_project_id, publisher_gid, "GROUP", "PROJECT_OWNER")
add_member_to_project(datazone, domain_id, patient_research_project_id, admin_uid, "USER", "PROJECT_OWNER")

## Data Management - Data Producer 

### 1. Varify Resources

This step can be done in AWS S3 console - varify the existance of 3 buckets 
1. patient-data-{suffix}
2. omics-data-{suffix}
3. images-data-{suffix}


### 2. Give patient_cohort project role access to data source bucket

In [ ]:
import re 
import json
s3 = boto3.client('s3')
iam = boto3.client('iam')

patient_cohort_project_role_arn = get_project_role_arn(datazone, domain_id, patient_cohort_project_id)
patient_cohort_project_role_name= patient_cohort_project_role_arn.split('/')[1]

print("patient_cohort_project_role_name", patient_cohort_project_role_name)

# List all buckets and filter by pattern
# there are 3 buckets provided by the workshop
# images-data-suffix
# omics-data-suffix
# patient-data-suffix
response = s3.list_buckets()
pattern = r'images-data-*'  # Your regex pattern

# find the suffix
suffix = None
for b in response['Buckets']: 
    if re.match(pattern, b['Name']):
        m = re.search(r'(?<=images-data-)\w+',b['Name'])
        suffix = m[0]
        break

policy_document = """{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject",
        "s3:ListBucket"
      ],
      "Resource": [
        "arn:aws:s3:::images-data-{suffix}",
        "arn:aws:s3:::images-data-{suffix}/*",
        "arn:aws:s3:::omics-data-{suffix}",
        "arn:aws:s3:::omics-data-{suffix}/*",
        "arn:aws:s3:::patient-data-{suffix}",
        "arn:aws:s3:::patient-data-{suffix}/*"
      ]
    }
  ]
}""".replace("{suffix}", suffix)

# attach policy to the project role 
iam.put_role_policy(RoleName=patient_cohort_project_role_name, 
                    PolicyName='sagemaker-clinical-data-repository-read-access',
                    PolicyDocument=policy_document)

# used to register location in lakeformation
data_buckets = [f"patients-data-{suffix}", f"omics-data-{suffix}", f"images-data-{suffix}"]


### 3. Run the crawlers in AWS Glue

In [ ]:
glue = boto3.client('glue')

crawler_names = ['patients_crawler', 'omics_crawler', 'images_crawler']
# Start the crawlers
for c in crawler_names:
    response = glue.start_crawler(Name=c)
    print(f"Crawler started: {response}")


In [ ]:
for c in crawler_names:
    response = glue.get_crawler(Name=c)
    print(f"Crawler state: {response['Crawler']['State']}")

### 4. Configure AWS Lakeformation 

#### 1. Add WSParticipant role as the Datalake administrator



In [ ]:
lakeformation = boto3.client('lakeformation')

# Get current settings
current_settings = lakeformation.get_data_lake_settings()
current_admins = current_settings['DataLakeSettings'].get('DataLakeAdmins', [])

participant_role_arn = boto3.client('iam').get_role(RoleName='WSParticipantRole')['Role']['Arn']
print(participant_role_arn)
# Add new admin
new_admin = {
    'DataLakePrincipalIdentifier': participant_role_arn
}
current_admins.append(new_admin)

# Update settings
lakeformation.put_data_lake_settings(
    DataLakeSettings={
        'DataLakeAdmins': current_admins
    }
)

#### 2. Grant project_role access to all patients, imaging, omics data tables in the data catalog


In [ ]:
lakeformation = boto3.client('lakeformation')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(account_id)

db_names = ['patients_data', 'omics_data', 'images_data']

for db_name in db_names:
    # Grant permissions to all tables in database
    response = lakeformation.grant_permissions(
        Principal={
            'DataLakePrincipalIdentifier': patient_cohort_project_role_arn
        },
        Resource={
            'Table': {
                'CatalogId': account_id,  
                'DatabaseName': db_name,
                'TableWildcard': {}  # This grants access to all tables
            }
        },
        Permissions=['SELECT', 'DESCRIBE']
    )
    print(response)

print(patient_cohort_project_role_arn)

####  3. Register datalake location

In [ ]:

for b in data_buckets: 
    lakeformation.register_resource(
        ResourceArn=f"arn:aws:s3:::{b}",
        UseServiceLinkedRole=True
    )

### Now proceed to Data Management - Data Producer /  Step 6 in the workshop and complete the workshop